# Cowpea IITA — PySpark

This notebook analyses the cowpea dataset using **PySpark only** (no pandas). Intended for use on Databricks.

|                  |                                               |
| ---------------- | --------------------------------------------- |
| **Crop**         | Cowpea (_Vigna unguiculata_)                  |
| **Locations**    | Ibadan, Ikenne, Kano (Nigeria)                |
| **Years**        | 2020–2022                                     |
| **Measurements** | 3,360                                         |
| **Genotypes**    | 112 (98 with SNP markers)                     |
| **Tables**       | `cowpea_iita_measurements`, `cowpea_iita_snp` |


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.connect.session import SparkSession

CATALOG = "open_jii_data_hackathon.default"

spark = SparkSession.getActiveSession()
assert spark is not None, "This notebook must be run on Databricks"

ms = spark.table(f"{CATALOG}.cowpea_iita_measurements")
snp = spark.table(f"{CATALOG}.cowpea_iita_snp")

In [ ]:
ms.printSchema()
ms.show(5)

In [ ]:
snp.printSchema()
snp.show(5, truncate=40)

## EDA


In [ ]:
print(f"Measurements: {ms.count():,}")
print(f"SNP markers: {snp.count():,}")
print(f"Genotypes in measurements: {ms.select('genotype').distinct().count()}")
print(f"Environments: {ms.select('env_id').distinct().count()}")

In [ ]:
# Counts by environment
ms.groupBy("env_id", "location", "year", "stress_condition").count().orderBy(
    "env_id"
).show()

In [ ]:
# Summary statistics for phenotype columns
phenotype_cols = [
    "relative_chlorophyll_content",
    "leaf_angle",
    "leaf_temp_differential",
    "lef",
    "npqt",
    "phi2",
    "phi_no",
    "phi_npq",
]

ms.select(phenotype_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()

In [ ]:
# Mean phenotype values by stress condition
ms.groupBy("stress_condition").agg(
    F.mean("phi2").alias("mean_phi2"),
    F.mean("lef").alias("mean_lef"),
    F.mean("npqt").alias("mean_npqt"),
    F.mean("relative_chlorophyll_content").alias("mean_spad"),
    F.mean("leaf_temp_differential").alias("mean_leaf_temp_diff"),
).show()

In [ ]:
# Mean phenotype values by location and stress
ms.groupBy("location", "stress_condition").agg(
    F.count("*").alias("n"),
    F.mean("phi2").alias("mean_phi2"),
    F.stddev("phi2").alias("std_phi2"),
    F.mean("lef").alias("mean_lef"),
    F.mean("npqt").alias("mean_npqt"),
).orderBy("location", "stress_condition").show()

In [ ]:
# Top 10 genotypes by mean Phi2
ms.groupBy("genotype").agg(
    F.count("*").alias("n"),
    F.mean("phi2").alias("mean_phi2"),
    F.mean("lef").alias("mean_lef"),
    F.mean("npqt").alias("mean_npqt"),
).orderBy(F.desc("mean_phi2")).show(10)

## Correlations


In [ ]:
# Pairwise Pearson correlations between phenotype traits
# Computed manually since pyspark.ml is not available on Databricks Connect
from itertools import combinations

ms_clean = ms.select(phenotype_cols).dropna()

corr_rows = []
for c1, c2 in combinations(phenotype_cols, 2):
    r = ms_clean.stat.corr(c1, c2)
    corr_rows.append((c1, c2, r))

print("Pairwise Pearson correlations:")
print(f"{'col1':>35s}  {'col2':>35s}  {'r':>7s}")
print("-" * 82)
for c1, c2, r in sorted(corr_rows, key=lambda x: abs(x[2]), reverse=True):
    print(f"{c1:>35s}  {c2:>35s}  {r:7.3f}")

## SNP data exploration


In [ ]:
# Markers per chromosome
snp.groupBy("chromosome").count().orderBy("chromosome").show()

In [ ]:
# Example: join mean phenotypes with SNP data
# Unpivot SNP table to long format for a specific marker
genotype_cols = [c for c in snp.columns if c.startswith("G")]

genotype_means = ms.groupBy("genotype").agg(F.mean("phi2").alias("mean_phi2"))

# Pick one marker as example
first_marker = snp.limit(1)
marker_id = first_marker.select("marker_id").head()[0]

# Unpivot genotype columns to rows
stack_expr = ", ".join(f"'{c}', `{c}`" for c in genotype_cols)
snp_long = first_marker.selectExpr(
    "marker_id",
    f"stack({len(genotype_cols)}, {stack_expr}) as (genotype, allele)",
)

# Join with phenotype means
joined = snp_long.join(genotype_means, on="genotype")

print(f"Marker: {marker_id}")
joined.groupBy("allele").agg(
    F.count("*").alias("n"), F.mean("mean_phi2").alias("phi2_by_allele")
).show()

## Stress response per genotype

Pivot mean phi2 by stress condition for each genotype, then compute a tolerance delta (before_stress − during_stress).


In [ ]:
# Pivot: mean phi2 per genotype × stress_condition
stress_pivot = ms.groupBy("genotype").pivot("stress_condition").agg(F.mean("phi2"))

stress_pivot.show(5)

# Compute delta = before - during
stress_delta = stress_pivot.withColumn(
    "delta_phi2",
    F.col("`Before Stress`") - F.col("`During Stress`"),
).filter(F.col("delta_phi2").isNotNull())

print("=== Top 10 most TOLERANT genotypes (smallest drop in phi2) ===")
stress_delta.orderBy("delta_phi2").show(10)

print("=== Top 10 most SENSITIVE genotypes (largest drop in phi2) ===")
stress_delta.orderBy(F.desc("delta_phi2")).show(10)

## Energy partitioning

phi2 + phi_npq + phi_no should sum to approximately 1. Check the balance by stress condition and by location × stress.


In [ ]:
# Energy partitioning: phi2 + phi_npq + phi_no ≈ 1
energy = ms.withColumn("energy_sum", F.col("phi2") + F.col("phi_npq") + F.col("phi_no"))

print("=== Mean energy fractions by stress_condition ===")
energy.groupBy("stress_condition").agg(
    F.mean("phi2").alias("mean_phi2"),
    F.mean("phi_npq").alias("mean_phi_npq"),
    F.mean("phi_no").alias("mean_phi_no"),
    F.mean("energy_sum").alias("mean_sum"),
    F.count("*").alias("n"),
).show()

print("=== Mean energy fractions by location × stress_condition ===")
energy.groupBy("location", "stress_condition").agg(
    F.mean("phi2").alias("mean_phi2"),
    F.mean("phi_npq").alias("mean_phi_npq"),
    F.mean("phi_no").alias("mean_phi_no"),
    F.mean("energy_sum").alias("mean_sum"),
    F.count("*").alias("n"),
).orderBy("location", "stress_condition").show(truncate=False)

## Genome-wide marker scan

For every SNP marker, unpivot genotype columns, join with mean phi2, and compute effect size (max − min mean phi2 across alleles). Identify the top 20 markers with the largest allelic effect on phi2.


In [ ]:
# Genome-wide marker scan — all markers at once
# Reuse genotype_cols from earlier cell

# Genotype mean phi2
genotype_means = ms.groupBy("genotype").agg(F.mean("phi2").alias("mean_phi2"))

# Unpivot ALL markers: (marker_id, chromosome, genotype, allele)
stack_expr = ", ".join(f"'{c}', `{c}`" for c in genotype_cols)
snp_long_all = snp.selectExpr(
    "marker_id",
    "chromosome",
    f"stack({len(genotype_cols)}, {stack_expr}) as (genotype, allele)",
)

# Join with phenotype means
snp_pheno = snp_long_all.join(genotype_means, on="genotype")

# Mean phi2 per marker × allele
marker_allele_means = (
    snp_pheno.filter(F.col("allele").isNotNull())
    .groupBy("marker_id", "chromosome", "allele")
    .agg(
        F.mean("mean_phi2").alias("allele_mean_phi2"),
        F.count("*").alias("n"),
    )
)

# Effect size per marker: max - min across alleles
marker_effects = (
    marker_allele_means.groupBy("marker_id", "chromosome")
    .agg(
        F.max("allele_mean_phi2").alias("max_phi2"),
        F.min("allele_mean_phi2").alias("min_phi2"),
        F.countDistinct("allele").alias("n_alleles"),
    )
    .withColumn("effect_size", F.col("max_phi2") - F.col("min_phi2"))
)

print("=== Top 20 markers by effect size on phi2 ===")
marker_effects.orderBy(F.desc("effect_size")).show(20, truncate=False)

print("=== Marker counts per chromosome ===")
marker_effects.groupBy("chromosome").agg(
    F.count("*").alias("n_markers"),
    F.max("effect_size").alias("max_effect"),
    F.mean("effect_size").alias("mean_effect"),
).orderBy("chromosome").show()

print("=== Top marker per chromosome ===")
from pyspark.sql import Window

w = Window.partitionBy("chromosome").orderBy(F.desc("effect_size"))
(
    marker_effects.withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy(F.desc("effect_size"))
    .show(truncate=False)
)

## Genotype × Environment interaction

Compute mean phi2 per genotype × env_id, pivot to wide format, then measure G×E plasticity as the variance of phi2 across environments for each genotype.


In [ ]:
# Genotype × Environment interaction
from pyspark.sql import Window

# Mean phi2 per genotype × env_id
gxe = ms.groupBy("genotype", "env_id").agg(
    F.mean("phi2").alias("mean_phi2"),
    F.count("*").alias("n"),
)

# Pivot to wide format: rows=genotype, columns=env_id
gxe_wide = gxe.groupBy("genotype").pivot("env_id").agg(F.first("mean_phi2"))

print("=== Genotype × Environment matrix (first 20 genotypes) ===")
gxe_wide.show(20, truncate=False)

# G×E plasticity: variance of mean phi2 across environments per genotype
# Use the long-format table for this calculation
gxe_plasticity = gxe.groupBy("genotype").agg(
    F.variance("mean_phi2").alias("var_phi2_across_envs"),
    F.mean("mean_phi2").alias("overall_mean_phi2"),
    F.count("env_id").alias("n_envs"),
)

print("=== Top 10 most PLASTIC genotypes (highest G×E variance) ===")
gxe_plasticity.orderBy(F.desc("var_phi2_across_envs")).show(10)

print("=== Top 10 most STABLE genotypes (lowest G×E variance) ===")
gxe_plasticity.filter(F.col("var_phi2_across_envs").isNotNull()).orderBy(
    "var_phi2_across_envs"
).show(10)